In [2]:
import numpy as np
from scr.nb_helper_functions import to_chunks
from scr.nb import NB_predictions, adjust_nb_predictions, visualize_predictions
from scr.validation_helpers import get_train_sample, get_nb_model
import matplotlib.pyplot as plt
import joblib

## Split the data in half

In [3]:
samples = get_train_sample()  # all 260
images = np.array(samples["image"])
species = np.array(samples["species"])  # keep labels too

mid = len(images) // 2

np.save("images_A.npy", images[:mid])
np.save("images_B.npy", images[mid:])
np.save("species_A.npy", species[:mid])
np.save("species_B.npy", species[mid:])

Using the latest cached version of the dataset since shenandoah-macroinvertebrates/ept-bioassessment-dataset couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/wilsonbeima/.cache/huggingface/datasets/shenandoah-macroinvertebrates___ept-bioassessment-dataset/default/0.0.0/9823ae8352792361e66545658a29afc8abacd86c (last modified on Sun Mar 15 13:53:48 2026).


# ------------ RUN THIS -------------------

In [11]:
try:
    model = joblib.load("models/naive_bayes_model_2.pkl")
except FileNotFoundError:
    raise NotImplementedError("Please provide a model at models/naive_bayes_model.pkl")

In [ ]:
images_B = np.load("images_B.npy")
subimages_B, predictions_B, shape = NB_predictions(images_B, model, slice_factor=1)
corrected_B = adjust_nb_predictions(subimages_B, predictions_B, shape)
np.savez("corrected_B.npz", subimages=subimages_B, predictions=corrected_B)

## Recombine data

In [ ]:
data_A = np.load("corrected_A.npz")
data_B = np.load("corrected_B.npz")

all_subimages = np.concatenate([data_A["subimages"], data_B["subimages"]], axis=0)
all_predictions = np.concatenate([data_A["predictions"], data_B["predictions"]], axis=0)
all_species = np.concatenate([np.load("species_A.npy"), np.load("species_B.npy")])